# Music DNA on a Colab GPU

Runs the slow parts on a GPU: Demucs stems, the vocal, intonation, Eastern and CQT features, then trains the moment model variants.

**Before you start**
1. On your Mac: `python -m musicdna.experiments.colab_bundle` makes `research-output/music_dna_colab.zip`.
2. Upload it to Google Drive as `MyDrive/music-dna/music_dna_colab.zip`.
3. In Colab: **Runtime → Change runtime type → GPU**, then run the cells top to bottom.

Research only: the audio stays in your own Drive. Results are written back to `MyDrive/music-dna/results/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, zipfile
DRIVE = '/content/drive/MyDrive/music-dna'
WORK = '/content/music-dna'
if not os.path.isdir(f'{WORK}/musicdna'):
    with zipfile.ZipFile(f'{DRIVE}/music_dna_colab.zip') as bundle:
        bundle.extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Colab already has a CUDA build of PyTorch; this installs the rest.
!pip install -q -r requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

## 1 · Features

Each stage skips songs it has already done, so re-running after a disconnect carries on where it stopped.
Demucs picks the GPU by itself; expect roughly 10-20 s per song instead of ~3 min on the Mac's CPU.

In [ ]:
!python -u -m musicdna.corpus.feature_store --corpus genre_mix --stages stems vocal intonation eastern cqt --workers 2

## 2 · Train and evaluate

`full` vs `western` is the main comparison; the three `full_no_*` runs are the ablations.
Every variant is trained on artist-grouped folds and scored on held-out songs, with Eastern, Western and house songs reported separately.

In [ ]:
!python -u -m musicdna.experiments.moment_model --variants full western full_no_cnn full_no_clap full_no_features --device cuda

## 3 · Save results to Drive

The report, plus the new features so the Mac can use them without recomputing.

In [ ]:
os.makedirs(f'{DRIVE}/results', exist_ok=True)
!cp research-output/moment_model_report.json {DRIVE}/results/
with zipfile.ZipFile(f'{DRIVE}/results/genre_mix_features.zip', 'w', zipfile.ZIP_STORED) as out:
    for root, _, names in os.walk('data/genre_mix/features'):
        for name in names:
            out.write(os.path.join(root, name))
print('saved to', f'{DRIVE}/results')